# aLIGO Input Mode Cleaner — Interactive Design Tool

A focused design reference for the 3-mirror triangular ring cavity,
seeded with near-real aLIGO IMC parameters.

**Geometry:** MC1 (flat, input) · MC2 (curved) · MC3 (flat, output)  
**Purpose:** spatial mode filtering · RF sideband rejection · pointing stability

---

| Quantity | Symbol | Default |
|----------|--------|---------|
| Carrier wavelength | λ | 1064 nm |
| Mirror ROC convention | | positive = concave (standard Siegman) |
| Lengths | | metres unless labelled |
| Frequencies | | Hz / MHz |


In [ ]:
import sys, os

try:
    import micropip
    from pyodide.http import pyfetch

    await micropip.install(["scipy", "plotly", "anywidget", "ipywidgets"])

    if os.path.isfile("/drive/src/cavity.py"):
        sys.path.insert(0, "/drive")
    else:
        _raw = "https://raw.githubusercontent.com/rxa254/optical-cavity-tutorial/main"
        os.makedirs("/src", exist_ok=True)
        for _mod in ["__init__", "beams", "cavity", "design",
                     "hom", "ring", "sidebands", "stability"]:
            _r = await pyfetch(f"{_raw}/src/{_mod}.py")
            open(f"/src/{_mod}.py", "w").write(await _r.string())
        sys.path.insert(0, "/")

except ImportError:
    if "." not in sys.path:
        sys.path.insert(0, ".")

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display


## 1. Introduction

The **Input Mode Cleaner (IMC)** is a triangular ring cavity between the
pre-stabilized laser and the main interferometer. It serves three purposes:

1. **Spatial filtering** — only TEM₀₀ reaches the IFO; higher-order spatial modes are reflected
2. **Frequency reference** — above the IMC pole (~13 kHz for aLIGO), laser frequency noise is filtered
3. **Pointing stabilisation** — beam jitter is suppressed because the cavity only transmits the aligned mode

### Geometry

The IMC is an isosceles triangle: two flat coupling mirrors (MC1, MC3) close together
and one curved focusing mirror (MC2) at the far end of a long vacuum tube.

```
            MC2  (curved, Rc)
           /    \
        d₁₂      d₂₃
         /          \
       MC1 ─ d₃₁ ─ MC3
     (flat)         (flat)
```

**Ring FSR = c / L** where L = d₁₂ + d₂₃ + d₃₁ is the full round-trip length.

### Stability

With two flat mirrors (identity ABCD) and one curved mirror, the round-trip matrix trace simplifies exactly to:

$$\mathrm{Tr}(M) = 2\left(1 - \frac{L}{R_c}\right)$$

The cavity is stable when |Tr(M)/2| ≤ 1, which gives the single condition **R_c > L/2**.

Round-trip Gouy phase: ψ = arccos(Tr(M)/2) = arccos(1 − L/R_c).  
HOM resonance offsets: Δf_q = (q ψ/π) mod FSR.

### aLIGO IMC seed parameters

| Parameter | Value |
|-----------|-------|
| d₁₂ = d₂₃ (long arms) | 16.1 m |
| d₃₁ (short leg, between flats) | 0.6 m |
| R_c (MC2) | 27.24 m |
| Round-trip length L | 32.8 m |
| FSR | 9.14 MHz |
| Gouy phase ψ | 101.8° |
| R (MC1 = MC3) | 0.993 |
| R (MC2) | 0.99999 |
| Finesse | ≈ 450 |
| Linewidth (FWHM) | ≈ 20 kHz |

*Parameters are approximate; see LIGO-T970051 for exact values.*


## 2. Ring cavity eigenmode

The self-consistent Gaussian mode is found by solving the round-trip ABCD
eigenvalue equation C·q² + (D−A)·q − B = 0 at mirror M1.

Use the sliders to explore how the geometry sets the beam shape.  Note:

- **Waist location**: for the near-symmetric IMC (d₁₂ ≈ d₂₃), the waist sits inside the
  short d₃₁ leg — between the two flat coupling mirrors.
- **Beam at MC2**: larger than at the flat mirrors because MC2 is near the waist conjugate.
- **d₁₂ = d₂₃** is enforced (symmetric triangle); d₃₁ is the short flat-to-flat spacing.


In [ ]:
from src.ring import ring_3m_mode, ring_fsr
from src.beams import beam_caustic

_WAV = 1064e-9

_sl2 = dict(style={'description_width': '210px'}, layout=widgets.Layout(width='520px'))
_d12_s = widgets.FloatSlider(value=16.1, min=2.0,  max=25.0, step=0.1,
                              description='d₁₂ = d₂₃ (m, long arm)',
                              readout_format='.2f', **_sl2)
_d31_s = widgets.FloatSlider(value=0.60, min=0.10, max=3.0,  step=0.05,
                              description='d₃₁ (m, short leg)',
                              readout_format='.2f', **_sl2)
_Rc_s  = widgets.FloatLogSlider(value=27.24, base=10,
                                 min=np.log10(5), max=np.log10(500), step=0.005,
                                 description='Rc of MC2 (m)',
                                 readout_format='.2f', **_sl2)
_info2 = widgets.HTML()

_fig2 = go.FigureWidget(make_subplots(rows=1, cols=2,
                                       subplot_titles=['Beam caustic (unfolded round trip)',
                                                       'Stability  Tr(M)/2 vs R_c'],
                                       column_widths=[0.62, 0.38]))
# [0] lower caustic envelope (invisible, fill base)
_fig2.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='steelblue', width=0),
                             showlegend=False), row=1, col=1)
# [1] upper caustic (fills to [0])
_fig2.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='steelblue', width=1.5),
                             fill='tonexty', fillcolor='rgba(70,130,180,0.25)',
                             showlegend=False), row=1, col=1)
# [2] MC1 position marker
_fig2.add_trace(go.Scatter(x=[0,0], y=[], mode='lines',
                             line=dict(color='gray', width=1, dash='dash'),
                             showlegend=False), row=1, col=1)
# [3] MC2 position marker
_fig2.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='dimgray', width=1.5, dash='dash'),
                             showlegend=False), row=1, col=1)
# [4] MC3 position marker
_fig2.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='gray', width=1, dash='dash'),
                             showlegend=False), row=1, col=1)
# [5] Tr/2 stability curve
_fig2.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='navy', width=1.5),
                             showlegend=False), row=1, col=2)
# [6] stable upper boundary y=+1
_fig2.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='red', width=1, dash='dash'),
                             showlegend=False), row=1, col=2)
# [7] stable lower boundary y=-1
_fig2.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='red', width=1, dash='dash'),
                             showlegend=False), row=1, col=2)
# [8] current operating point
_fig2.add_trace(go.Scatter(x=[None], y=[None], mode='markers',
                             marker=dict(size=10), showlegend=False), row=1, col=2)

_fig2.update_xaxes(title_text='Position along cavity (m)', row=1, col=1)
_fig2.update_yaxes(title_text='Beam radius (mm)', row=1, col=1)
_fig2.update_xaxes(title_text='Rc (m)', type='log', row=1, col=2)
_fig2.update_yaxes(title_text='Tr(M)/2', range=[-2.5, 2.5], row=1, col=2)
_fig2.update_layout(height=460, margin=dict(t=50, b=40, l=60, r=20))

_Rc_plot = np.logspace(np.log10(5), np.log10(500), 400)


def _update2(_change=None):
    d12 = _d12_s.value
    d31 = _d31_s.value
    Rc  = _Rc_s.value
    d23 = d12

    m   = ring_3m_mode(d12, d23, d31, Rc, _WAV)
    L   = m['L']
    tr2 = m['Tr'] / 2
    tr_curve = (1.0 - L / _Rc_plot).tolist()

    with _fig2.batch_update():
        _fig2.data[5].x = _Rc_plot.tolist()
        _fig2.data[5].y = tr_curve
        _fig2.data[6].x = [5.0, 500.0];  _fig2.data[6].y = [1.0, 1.0]
        _fig2.data[7].x = [5.0, 500.0];  _fig2.data[7].y = [-1.0, -1.0]
        _fig2.data[8].x = [Rc];           _fig2.data[8].y = [tr2]
        _fig2.data[8].marker.color = 'steelblue' if m['stable'] else 'red'

        if m['stable']:
            N   = 500
            z1  = np.linspace(0, d12, N)
            z2  = np.linspace(0, d23, N)
            z3  = np.linspace(0, d31, N)
            w1a = beam_caustic(m['q1'],         z1, _WAV) * 1e3
            w2a = beam_caustic(m['q_after_M2'], z2, _WAV) * 1e3
            w3a = beam_caustic(m['q_at_M3'],    z3, _WAV) * 1e3

            z_all = np.concatenate([z1, d12 + z2, d12 + d23 + z3])
            w_all = np.concatenate([w1a, w2a, w3a])
            ymax  = float(np.max(w_all)) * 1.2

            _fig2.data[0].x = z_all.tolist(); _fig2.data[0].y = (-w_all).tolist()
            _fig2.data[1].x = z_all.tolist(); _fig2.data[1].y = w_all.tolist()
            _fig2.data[2].x = [0, 0];          _fig2.data[2].y = [-ymax, ymax]
            _fig2.data[3].x = [d12, d12];      _fig2.data[3].y = [-ymax, ymax]
            _fig2.data[4].x = [d12+d23, d12+d23]; _fig2.data[4].y = [-ymax, ymax]
        else:
            for i in range(5):
                _fig2.data[i].x = []; _fig2.data[i].y = []

    if m['stable']:
        fsr_hz = ring_fsr(L)
        dw = m['d_waist']
        if dw < 0:
            waist_str = f'{abs(dw):.3f} m before MC1 (inside d₃₁ leg)'
        elif dw <= d12:
            waist_str = f'{dw:.3f} m from MC1 (MC1→MC2 leg)'
        else:
            waist_str = 'beyond MC2'
        _info2.value = (
            '<table style="font-size:13px;border-collapse:collapse;margin-top:6px">'
            f'<tr>'
            f'<td style="padding:3px 18px"><b>L</b></td><td>{L:.3f} m</td>'
            f'<td style="padding:3px 18px"><b>FSR</b></td><td>{fsr_hz/1e6:.3f} MHz</td>'
            f'<td style="padding:3px 18px"><b>Gouy ψ</b></td><td>{m["gouy_deg"]:.1f}°</td>'
            f'</tr><tr>'
            f'<td style="padding:3px 18px"><b>w(MC1)</b></td><td>{m["w1"]*1e3:.3f} mm</td>'
            f'<td style="padding:3px 18px"><b>w(MC2)</b></td><td>{m["w2"]*1e3:.3f} mm</td>'
            f'<td style="padding:3px 18px"><b>w(MC3)</b></td><td>{m["w3"]*1e3:.3f} mm</td>'
            f'</tr><tr>'
            f'<td style="padding:3px 18px"><b>Waist w₀</b></td><td>{m["w0"]*1e3:.3f} mm</td>'
            f'<td style="padding:3px 18px"><b>z_R</b></td><td>{m["z_R"]:.2f} m</td>'
            f'<td style="padding:3px 18px"><b>Location</b></td><td>{waist_str}</td>'
            f'</tr>'
            '</table>'
        )
    else:
        _info2.value = (
            f'<p style="color:darkorange;font-size:14px">'
            f'<b>Unstable</b> — Tr(M)/2 = {tr2:.4f}. '
            f'Stability requires R_c &gt; L/2 = {m["L"]/2:.2f} m.</p>'
        )


for _w2 in (_d12_s, _d31_s, _Rc_s):
    _w2.observe(_update2, names='value')

display(widgets.VBox([_d12_s, _d31_s, _Rc_s, _info2, _fig2]))
_update2()


## 3. Cavity parameters

For a 3-mirror ring cavity (MC1 = MC3 coupling mirrors, MC2 high-reflector):

$$F = \frac{\pi\,\sqrt{r}}{1-r}, \quad r = \sqrt{R_1 R_2 R_3}$$

where r is the round-trip amplitude reflectivity and R_i are power reflectivities.

The coupling regime is set by the **input coupler** MC1 relative to the total round-trip loss from MC2 and MC3:
- **Critical** (maximum transmission): R₁ = R₂ R₃
- **Over-coupled** (R₁ < R₂R₃): more power reflected than lost per round trip
- **Under-coupled** (R₁ > R₂R₃): input coupler too lossy


In [ ]:
from src.ring import ring_3m_finesse, ring_fsr, ring_3m_mode
from src.cavity import linewidth

_sl3 = dict(style={'description_width': '220px'}, layout=widgets.Layout(width='530px'))
_Rmc1_s = widgets.FloatSlider(value=0.9930, min=0.900, max=0.9999, step=0.001,
                               description='R₁ = R₃  (MC1/MC3, coupling)',
                               readout_format='.4f', **_sl3)
_Rmc2_s = widgets.FloatSlider(value=0.99999, min=0.9990, max=0.999999, step=0.00001,
                               description='R₂  (MC2, curved HR)',
                               readout_format='.6f', **_sl3)
_d12b_s = widgets.FloatSlider(value=16.1, min=2.0, max=25.0, step=0.5,
                               description='d₁₂ = d₂₃ (m)',
                               readout_format='.1f', **_sl3)
_d31b_s = widgets.FloatSlider(value=0.60, min=0.10, max=3.0, step=0.1,
                               description='d₃₁ (m, short leg)',
                               readout_format='.2f', **_sl3)
_Rcb_s  = widgets.FloatSlider(value=27.24, min=15.0, max=100.0, step=0.5,
                               description='Rc of MC2 (m)',
                               readout_format='.2f', **_sl3)
_info3  = widgets.HTML()


def _fmt3(hz):
    if hz >= 1e6: return f'{hz/1e6:.3g} MHz'
    if hz >= 1e3: return f'{hz/1e3:.3g} kHz'
    return f'{hz:.3g} Hz'


def _update3(_change=None):
    R1 = _Rmc1_s.value
    R2 = _Rmc2_s.value
    d12 = _d12b_s.value
    d31 = _d31b_s.value
    Rc  = _Rcb_s.value
    L   = 2 * d12 + d31

    F   = ring_3m_finesse(R1, R2)
    fsr_hz = ring_fsr(L)
    lw  = linewidth(F, fsr_hz)
    # Power buildup (input side): (1-R1) / (1 - r)^2  where r = sqrt(R1*R2*R3)
    import numpy as np
    r   = np.sqrt(R1 * R2 * R1)
    B   = (1.0 - R1) / (1.0 - r) ** 2

    # Coupling: compare r_in = sqrt(R1) to r_loss = sqrt(R2*R3) = sqrt(R2*R1)
    r_in   = np.sqrt(R1)
    r_loss = np.sqrt(R2 * R1)   # R3 = R1
    tol = 0.002
    if abs(r_in - r_loss) / max(r_in, r_loss) < tol:
        coup = 'critical'
    elif r_in < r_loss:
        coup = 'over'
    else:
        coup = 'under'

    coup_html = {
        'critical': '<b style="color:green">critically coupled</b>',
        'over':     '<b style="color:royalblue">over-coupled</b> (r₁ &lt; r₂r₃)',
        'under':    '<b style="color:darkorange">under-coupled</b> (r₁ &gt; r₂r₃)',
    }[coup]

    _info3.value = (
        '<table style="font-size:14px;border-collapse:collapse;margin-top:8px">'
        f'<tr><td style="padding:4px 24px"><b>Finesse</b></td><td>{F:.0f}</td>'
        f'    <td style="padding:4px 24px"><b>FSR</b></td><td>{_fmt3(fsr_hz)}</td></tr>'
        f'<tr><td style="padding:4px 24px"><b>Linewidth</b></td><td>{_fmt3(lw)}</td>'
        f'    <td style="padding:4px 24px"><b>Peak buildup</b></td><td>{B:.0f}×</td></tr>'
        f'<tr><td style="padding:4px 24px"><b>Round trip L</b></td><td>{L:.2f} m</td>'
        f'    <td style="padding:4px 24px"><b>Coupling</b></td><td>{coup_html}</td></tr>'
        '</table>'
    )


for _w3 in (_Rmc1_s, _Rmc2_s, _d12b_s, _d31b_s, _Rcb_s):
    _w3.observe(_update3, names='value')

display(widgets.VBox([_Rmc1_s, _Rmc2_s, _d12b_s, _d31b_s, _Rcb_s, _info3]))
_update3()


## 4. Higher-order modes & sideband placement

The IMC must keep the RF modulation sidebands **off** any HOM resonance so
that the mode-cleaner does not partially transmit the sidebands.

HOM resonance frequencies for the ring cavity:

$$\Delta f_q = \frac{q\,\psi}{\pi}\,\text{FSR} \pmod{\text{FSR}}, \quad \psi = \arccos\!\left(1 - \frac{L}{R_c}\right)$$

The aLIGO IMC uses a modulation frequency of **f_mod = 24.08 MHz ≈ 2.64 × FSR**.
The sidebands at ±1f land **between** HOM groups — check this with the widget.

Color convention: **blue** = TEM₀₀, **green** = HOM groups, **orange** = RF sidebands.


In [ ]:
from src.ring import ring_hom_offsets, ring_fsr, ring_3m_mode
from src.hom import tem00_spectrum

_sl4 = dict(style={'description_width': '220px'}, layout=widgets.Layout(width='530px'))
_Rc4_s  = widgets.FloatSlider(value=27.24, min=15.0, max=100.0, step=0.5,
                               description='Rc of MC2 (m)',
                               readout_format='.2f', **_sl4)
_d124_s = widgets.FloatSlider(value=16.1, min=2.0, max=25.0, step=0.5,
                               description='d₁₂ = d₂₃ (m)',
                               readout_format='.1f', **_sl4)
_d314_s = widgets.FloatSlider(value=0.60, min=0.10, max=3.0, step=0.1,
                               description='d₃₁ (m)',
                               readout_format='.2f', **_sl4)
_F4_s   = widgets.FloatLogSlider(value=450, base=10, min=1, max=5, step=0.01,
                                  description='Finesse',
                                  readout_format='.0f', **_sl4)
_fmod4  = widgets.FloatSlider(value=24.08, min=1.0, max=80.0, step=0.1,
                               description='f_mod (MHz)',
                               readout_format='.2f', **_sl4)
_maxq4  = widgets.IntSlider(value=6, min=1, max=12, step=1,
                             description='Max mode order', **_sl4)
_beta4  = widgets.FloatSlider(value=0.3, min=0.01, max=2.0, step=0.01,
                               description='Mod. depth β',
                               readout_format='.2f', **_sl4)
_log4   = widgets.Checkbox(value=False, description='Log y-axis',
                            layout=widgets.Layout(width='auto'))
_warn4  = widgets.HTML()

_fig4 = go.FigureWidget()
# [0] TEM00 spectrum
_fig4.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='steelblue', width=1.5),
                             fill='tozeroy', fillcolor='rgba(70,130,180,0.15)',
                             name='TEM₀₀'))
# [1] even-m HOMs (m = 0, 2, 4 …) — green
_fig4.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='mediumseagreen', width=1.2),
                             fill='tozeroy', fillcolor='rgba(60,179,113,0.12)',
                             name='HOMs (even m)'))
# [2] odd-m HOMs (m = 1, 3, 5 …) — orchid (odd-bounce parity shift)
_fig4.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='orchid', width=1.2),
                             fill='tozeroy', fillcolor='rgba(218,112,214,0.12)',
                             name='HOMs (odd m)'))
# [3] ±1f sideband lines
_fig4.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='darkorange', width=1.5, dash='dash'),
                             name='±1f sideband'))
# [4] ±2f/3f sideband lines
_fig4.add_trace(go.Scatter(x=[], y=[], mode='lines',
                             line=dict(color='darkorange', width=0.8, dash='dot'),
                             opacity=0.55, name='±2f/3f sideband'))

_fig4.update_xaxes(title_text='Frequency offset from carrier (MHz)')
_fig4.update_yaxes(title_text='Normalised amplitude', range=[0, 1.15])
_fig4.update_layout(height=420, margin=dict(t=30, b=40, l=60, r=20), legend=dict(x=0.01, y=0.98))


def _update4(_change=None):
    Rc   = _Rc4_s.value
    d12  = _d124_s.value
    d31  = _d314_s.value
    F    = _F4_s.value
    fmod = _fmod4.value * 1e6
    maxq = _maxq4.value
    L    = 2 * d12 + d31
    fsr  = ring_fsr(L)
    lw   = fsr / F
    beta = _beta4.value

    # Frequency axis: 2 FSR span
    N = 8000
    freq = np.linspace(-fsr, fsr, N)

    # TEM00 spectrum
    tem00 = tem00_spectrum(freq, fsr, F)

    # HOM spectrum split by horizontal parity (odd-bounce effect)
    offsets = ring_hom_offsets(maxq, fsr, Rc, L)
    hw_sq   = (lw / 2.0) ** 2
    even_x, even_y = [], []   # m even  (green)
    odd_x,  odd_y  = [], []   # m odd   (orchid)
    for (m, n, df, label) in offsets:
        for k in [-1, 0, 1]:
            f0   = k * fsr + df
            peak = hw_sq / ((freq - f0) ** 2 + hw_sq)
            if m % 2 == 0:
                even_x.extend(freq.tolist()); even_y.extend(peak.tolist())
                even_x.append(float('nan')); even_y.append(float('nan'))
            else:
                odd_x.extend(freq.tolist());  odd_y.extend(peak.tolist())
                odd_x.append(float('nan'));   odd_y.append(float('nan'))

    # RF sideband lines (±1f solid, ±2f/3f dotted)
    sb_x, sb_y = [], []
    for sign in [+1, -1]:
        f_sb = sign * fmod
        sb_x += [f_sb, f_sb, float('nan')]
        sb_y += [0, 1.1, float('nan')]
    from scipy.special import jv as _jv4
    j1 = max(abs(float(_jv4(1, beta))), 1e-10)
    sb2_x, sb2_y = [], []
    for h in [2, 3]:
        j_h   = max(abs(float(_jv4(h, beta))), 1e-10)
        h_top = min(0.85 * j_h / j1, 0.85)   # scale peak height by relative amplitude
        for sign in [+1, -1]:
            f_sb = sign * h * fmod
            if abs(f_sb) <= fsr:
                sb2_x += [f_sb, f_sb, float('nan')]
                sb2_y += [0, h_top, float('nan')]

    with _fig4.batch_update():
        _fig4.data[0].x = (freq / 1e6).tolist(); _fig4.data[0].y = tem00.tolist()
        _fig4.data[1].x = [f/1e6 for f in even_x]; _fig4.data[1].y = even_y
        _fig4.data[2].x = [f/1e6 for f in odd_x];  _fig4.data[2].y = odd_y
        _fig4.data[3].x = [f/1e6 for f in sb_x];   _fig4.data[3].y = sb_y
        _fig4.data[4].x = [f/1e6 for f in sb2_x];  _fig4.data[4].y = sb2_y
    if _log4.value:
        _fig4.update_layout(yaxis=dict(type='log',    range=[-5, 0.1]))
    else:
        _fig4.update_layout(yaxis=dict(type='linear', range=[0, 1.15]))

    # Accidental resonance check: ±1f/2f/3f sidebands within 10 lw of any HOM
    # Threshold scaled by J_1/J_h so weaker sidebands need to be much closer to warn
    warnings = []
    for h in [1, 2, 3]:
        j_h_warn = max(abs(float(_jv4(h, beta))), 1e-10)
        thresh   = 10.0 * lw * j_h_warn / j1   # effective threshold (linewidths)
        for sign in [+1, -1]:
            f_sb = (sign * h * fmod) % fsr
            if f_sb > fsr / 2:
                f_sb -= fsr
            for (m, n, df, label) in offsets:
                df_centered = df if df < fsr / 2 else df - fsr
                sep = abs(f_sb - df_centered)
                if sep < thresh:
                    tag = f'{sign:+d}{h}f' if h > 1 else f'{sign:+d}f'
                    warnings.append(
                        f'⚠ {tag} sideband ({sign*h*fmod/1e6:.2f} MHz) is within '
                        f'{sep/lw:.1f} lw of HOM {label}'
                    )

    if warnings:
        _warn4.value = '<br>'.join(f'<span style="color:red">{w}</span>' for w in warnings)
    else:
        _warn4.value = f'<span style="color:green">✓ No sideband–HOM collisions within 3 linewidths</span>'


for _w4 in (_Rc4_s, _d124_s, _d314_s, _F4_s, _fmod4, _maxq4, _beta4, _log4):
    _w4.observe(_update4, names='value')

display(widgets.VBox([_Rc4_s, _d124_s, _d314_s, _F4_s, _fmod4, _maxq4, _beta4, _log4, _warn4, _fig4]))
_update4()

## 5. Multi-objective design optimizer

Every IMC design is a compromise between five competing objectives:

| Objective | Meaning | Score = 1 when |
|-----------|---------|----------------|
| **Stability** | Distance from stability boundary | Rc ≫ L/2 |
| **HOM avoidance** | Sideband–HOM separation | ≥ 10 linewidths from any HOM |
| **PDH slope** | Sideband placement | f_mod ≥ 10 × linewidth |
| **Finesse** | Mirror quality | F ≥ 2000 |
| **Beam waist** | Mode size at M1 | waist in [0.5, 8] mm |

Use the weight sliders to emphasise what matters most for your design, then click **Optimize** to run a greedy coordinate-descent search.

In [ ]:
from src.ring import (
    ring_design_scores, ring_optimize, RING_OBJECTIVES,
    ring_3m_finesse, ring_fsr, ring_3m_mode,
)

_N5     = len(RING_OBJECTIVES)
_lbls5  = [lbl for _, lbl in RING_OBJECTIVES]
_keys5  = [k   for k, _   in RING_OBJECTIVES]
_angles5 = [n / _N5 * 360 for n in range(_N5)] + [0]

_sl5p = dict(style={'description_width': '200px'}, layout=widgets.Layout(width='480px'))
_sl5w = dict(style={'description_width': '160px'}, layout=widgets.Layout(width='380px'))

# ── parameter sliders ────────────────────────────────────────────────────
_d125  = widgets.FloatSlider(value=16.1,  min=2.0,   max=25.0,  step=0.5,
                              description='d₁₂ = d₂₃ (m)', readout_format='.1f', **_sl5p)
_d315  = widgets.FloatSlider(value=0.60,  min=0.1,   max=3.0,   step=0.05,
                              description='d₃₁ (m)',           readout_format='.2f', **_sl5p)
_Rc5   = widgets.FloatSlider(value=27.24, min=15.0,  max=100.0, step=0.5,
                              description='Rc of MC2 (m)',      readout_format='.2f', **_sl5p)
_R5    = widgets.FloatSlider(value=0.9978, min=0.90, max=0.9999, step=0.0001,
                              description='R (coupling mirrors)', readout_format='.4f', **_sl5p)
_fmod5 = widgets.FloatSlider(value=24.08, min=1.0,   max=80.0,  step=0.1,
                              description='f_mod (MHz)',         readout_format='.2f', **_sl5p)
_beta5 = widgets.FloatSlider(value=0.3,   min=0.01,  max=2.0,   step=0.01,
                              description='Mod. depth β',        readout_format='.2f', **_sl5p)

# ── weight sliders ───────────────────────────────────────────────────────
_WS5 = {k: widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.1,
                                description=f'w: {lbl}', readout_format='.1f', **_sl5w)
        for k, lbl in RING_OBJECTIVES}

# ── optimizer controls ───────────────────────────────────────────────────
_btn5  = widgets.Button(description='Optimize', button_style='primary',
                         layout=widgets.Layout(width='120px', height='32px'))
_cyc5  = widgets.IntSlider(value=3, min=1, max=6, step=1,
                            description='Cycles',
                            style={'description_width': '60px'},
                            layout=widgets.Layout(width='240px'))
_fix_fsr5 = widgets.Checkbox(value=True, description='Fix FSR (IFO constraint)',
                              layout=widgets.Layout(width='auto'))
_log5  = widgets.HTML()
_info5 = widgets.HTML()

# ── figure: radar + bar ──────────────────────────────────────────────────
_fig5 = go.FigureWidget(make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'polar'}, {'type': 'xy'}]],
    subplot_titles=['Score radar', 'Individual scores']))

# [0] reference ring
_fig5.add_trace(go.Scatterpolar(
    r=[1.0] * (_N5 + 1), theta=_angles5, mode='lines',
    line=dict(color='lightgray', width=1, dash='dash'),
    showlegend=False), row=1, col=1)
# [1] score polygon (dynamic)
_fig5.add_trace(go.Scatterpolar(
    r=[0] * (_N5 + 1), theta=_angles5, mode='lines+markers',
    fill='toself', fillcolor='rgba(70,130,180,0.25)',
    line=dict(color='steelblue', width=2), showlegend=False), row=1, col=1)
# [2] bar chart (dynamic)
_fig5.add_trace(go.Bar(
    x=_lbls5, y=[0] * _N5,
    marker_color=['steelblue'] * _N5, showlegend=False), row=1, col=2)

_fig5.update_polars(
    radialaxis=dict(range=[0, 1], tickvals=[0.25, 0.5, 0.75, 1.0]),
    angularaxis=dict(tickvals=_angles5[:-1], ticktext=_lbls5))
_fig5.update_yaxes(range=[0, 1.15], title_text='Score', row=1, col=2)
_fig5.update_layout(height=440, margin=dict(t=50, b=40, l=20, r=20))

_updating5 = [False]   # guard against slider-cascade during optimizer update


def _update5(_=None):
    if _updating5[0]:
        return
    d12 = _d125.value; d31 = _d315.value; Rc = _Rc5.value
    R   = _R5.value;   fmod = _fmod5.value
    weights = {k: _WS5[k].value for k in _keys5}

    s    = ring_design_scores(d12, d31, Rc, R, fmod, weights=weights, beta=_beta5.value)
    vals = [s[k] for k in _keys5]
    vplot = vals + [vals[0]]

    bar_colors = ['steelblue' if v > 0.7 else ('darkorange' if v > 0.4 else 'crimson')
                  for v in vals]
    with _fig5.batch_update():
        _fig5.data[1].r = vplot
        _fig5.data[2].y = vals
        _fig5.data[2].marker.color = bar_colors
        _fig5.data[2].text = [f'{v:.2f}' for v in vals]
        _fig5.data[2].textposition = 'outside'
    _fig5.update_polars(radialaxis_title_text=f'Total: {s["total"]:.2f}')

    # Info table
    L    = 2 * d12 + d31
    F    = ring_3m_finesse(R, 0.99999)
    fsr  = ring_fsr(L)
    lw   = fsr / F
    res  = ring_3m_mode(d12, d12, d31, Rc)
    w0s  = f'{res["w0"]*1e3:.2f} mm' if res['stable'] else '—'
    stab = ('<b style="color:green">stable</b>' if res['stable']
            else '<b style="color:red">UNSTABLE</b>')
    _info5.value = (
        '<table style="font-size:13px;border-collapse:collapse;margin-top:6px"><tr>'
        f'<td style="padding:3px 18px"><b>L</b></td><td>{L:.2f} m</td>'
        f'<td style="padding:3px 18px"><b>FSR</b></td><td>{fsr/1e6:.3f} MHz</td>'
        f'<td style="padding:3px 18px"><b>Finesse</b></td><td>{F:.0f}</td>'
        f'<td style="padding:3px 18px"><b>Linewidth</b></td><td>{lw/1e3:.2f} kHz</td>'
        f'<td style="padding:3px 18px"><b>w₀ (M1)</b></td><td>{w0s}</td>'
        f'<td style="padding:3px 18px"><b>Stability</b></td><td>{stab}</td>'
        '</tr></table>'
    )


def _on_opt5(_):
    _btn5.description = 'Optimizing…'
    _btn5.disabled = True
    weights = {k: _WS5[k].value for k in _keys5}
    opt = ring_optimize(
        _d125.value, _d315.value, _Rc5.value, _R5.value, _fmod5.value,
        weights=weights, n_cycles=int(_cyc5.value),
        fix_fsr=_fix_fsr5.value, beta=_beta5.value,
    )
    r = opt['result']
    _updating5[0] = True
    _d125.value  = r['d12_m']
    _d315.value  = r['d31_m']
    _Rc5.value   = r['Rc_m']
    _R5.value    = r['R']
    _fmod5.value = r['f_mod_mhz']
    _updating5[0] = False
    _update5()
    _log5.value = (
        '<pre style="font-size:11px;margin-top:10px;padding:6px;'
        'background:#f8f8f8;border:1px solid #ddd;border-radius:4px">'
        + '\n'.join(opt['history'])
        + '</pre>'
    )
    _btn5.description = 'Optimize'
    _btn5.disabled = False


_btn5.on_click(_on_opt5)
for _w5 in [_d125, _d315, _Rc5, _R5, _fmod5, _beta5] + list(_WS5.values()):
    _w5.observe(_update5, names='value')

display(widgets.VBox([
    widgets.HBox([
        widgets.VBox([_d125, _d315, _Rc5, _R5, _fmod5, _beta5]),
        widgets.VBox([_WS5[k] for k in _keys5]),
    ]),
    widgets.HBox([_btn5, _cyc5, _fix_fsr5]),
    _info5,
    _fig5,
    _log5,
]))
_update5()